In [ ]:
import re
import pandas as pd
from collections import defaultdict

def analyze_ns3_output(log_file_path):
    """
    Parses the ns3 log file to track packet paths, timings, and tags.

    Args:
        log_file_path (str): The path to the ns3 log file.

    Returns:
        pandas.DataFrame: A DataFrame with packet information, paths, and timings.
    """
    packets_info = {}
    packet_hops = defaultdict(list)

    # Regex to parse FlowInfo and LinkTx lines, now capturing more data
    flow_info_re = re.compile(r"FlowInfo, .*PktUID: (\d+), OriginalSrc: (\d+), FinalDst: (\d+), Sport: (\d+), FlowTag: (\d+)")
    flow_info_1_re = re.compile(r"FlowInfo-1, PktUID: (\d+)")
    link_tx_re = re.compile(r"LinkTx, Timestamp: (\d+)ns, .*PktUID: (\d+), LinkSrc: (\d+)")

    with open(log_file_path, 'r') as f:
        for line in f:
            # Match FlowInfo lines to get packet metadata
            flow_match = flow_info_re.search(line)
            if flow_match:
                pkt_uid, src, dst, port, tag = map(int, flow_match.groups())
                if pkt_uid not in packets_info:
                    packets_info[pkt_uid] = {'OriginalSrc': src, 'FinalDst': dst, 'Tag': tag, 'Port': port}
                continue

            # Match FlowInfo-1 lines (no OriginalSrc/Tag)
            flow_1_match = flow_info_1_re.search(line)
            if flow_1_match:
                pkt_uid = int(flow_1_match.group(1))
                if pkt_uid not in packets_info:
                    packets_info[pkt_uid] = {'OriginalSrc': None, 'FinalDst': None, 'Tag': None}
                continue

            # Match LinkTx lines to build the packet path with timestamps
            link_match = link_tx_re.search(line)
            if link_match:
                timestamp, pkt_uid, link_src = map(int, link_match.groups())
                # Append (source, timestamp) to the packet's hop list
                current_hops = packet_hops[pkt_uid]
                if not current_hops or current_hops[-1][0] != link_src:
                    current_hops.append((link_src, timestamp))

    # Combine the parsed data into a list of dictionaries
    records = []
    for uid, hops in packet_hops.items():
        info = packets_info.get(uid, {})
        path_nodes = [hop[0] for hop in hops]
        
        start_time = hops[0][1] if hops else 0
        end_time = hops[-1][1] if hops else 0
        delay = end_time - start_time if hops else 0
        
        original_src = info.get('OriginalSrc')
        if original_src is None and path_nodes:
            original_src = path_nodes[0]
        
        records.append({
            'PktUID': uid,
            'Tag': info.get('Tag'),
            'Port' : info.get('Port'),
            'OriginalSrc': original_src,
            'FinalDst': info.get('FinalDst'),
            'StartTime (ns)': start_time,
            'EndTime (ns)': end_time,
            'Delay (ns)': delay,
            'Path': ' -> '.join(map(str, path_nodes)),
            'Path_List': path_nodes
        })

    # Create and return a DataFrame
    df = pd.DataFrame(records)
    if not df.empty:
        df = df.sort_values(by=['OriginalSrc', 'PktUID']).reset_index(drop=True)
    return df

# --- Main execution ---
# Path to your ns3 log file
ns3_log_path = "/home/xavid/feina/astra-sim/upc/text_ns3.txt"

# Analyze the log file
packet_tracking_df = analyze_ns3_output(ns3_log_path)

# Display the paths for packets originating from each source node
if not packet_tracking_df.empty:
    for src_node in sorted(packet_tracking_df['OriginalSrc'].dropna().unique()):
        print(f"--- Paths from OriginalSrc: {src_node} ---")
        src_df = packet_tracking_df[packet_tracking_df['OriginalSrc'] == src_node]
        
        # Group by path to show unique paths and the packets that took them
        path_groups = src_df.groupby('Path')['PktUID'].apply(list)
        for path, uids in path_groups.items():
            print(f"  Path: {path}")
            print(f"    PktUIDs: {uids}\n")
else:
    print("No packet information found in the log file.")



In [ ]:
if not packet_tracking_df.empty:
    # Get unique pairs of OriginalSrc and FinalDst
    unique_pairs = packet_tracking_df[['OriginalSrc', 'FinalDst']].dropna().drop_duplicates()
    
    for index, row in unique_pairs.iterrows():
        src = int(row['OriginalSrc'])
        dst = int(row['FinalDst'])
        
        print(f"--- Path counts from OriginalSrc: {src} to FinalDst: {dst} ---")
        
        # Filter for the specific src-dst pair
        filtered_df = packet_tracking_df[
            (packet_tracking_df['OriginalSrc'] == src) & 
            (packet_tracking_df['FinalDst'] == dst)
        ]
        
        # Count the different paths taken
        path_counts = filtered_df['Path_List'].value_counts()
        
        if not path_counts.empty:
            print(path_counts)
        else:
            print("No paths found for this pair.")
        print("-" * 50)
else:
    print("Packet tracking DataFrame is empty.")

In [ ]:
packet_tracking_df[(packet_tracking_df['OriginalSrc'] == 2) ].head()

In [ ]:
if not packet_tracking_df.empty:
    # Get unique pairs of OriginalSrc and FinalDst
    unique_pairs = packet_tracking_df[['OriginalSrc', 'FinalDst']].dropna().drop_duplicates()
    
    for index, row in unique_pairs.iterrows():
        src = int(row['OriginalSrc'])
        dst = int(row['FinalDst'])
        
        print(f"--- Sequential Path Analysis for OriginalSrc: {src} to FinalDst: {dst} ---")
        
        # Filter for the specific src-dst pair, already sorted by PktUID
        pair_df = packet_tracking_df[
            (packet_tracking_df['OriginalSrc'] == src) & 
            (packet_tracking_df['FinalDst'] == dst)
        ].copy()
        
        if not pair_df.empty:
            # Identify consecutive groups of packets taking the same path
            # A new group starts when the path changes from the previous packet
            pair_df['PathStr'] = pair_df['Path_List'].astype(str)
            pair_df['PathGroup'] = (pair_df['PathStr'] != pair_df['PathStr'].shift()).cumsum()

            # Group by these sequential path groups and aggregate the results
            grouped_paths = pair_df.groupby('PathGroup').agg(
                Path=('Path', 'first'),
                PacketCount=('PktUID', 'count'),
                GroupStartTime=('StartTime (ns)', 'min'),
                GroupEndTime=('EndTime (ns)', 'max')
            )
            
            # Calculate duration for each group
            grouped_paths['GroupDuration (ns)'] = grouped_paths['GroupEndTime'] - grouped_paths['GroupStartTime']
            
            # Display the results for each sequential group
            for i, group_row in grouped_paths.iterrows():
                print(f"\n  Path: {group_row['Path']}")
                print(f"    Sequential Packet Count: {group_row['PacketCount']}")
                print(f"    Group Start Time:        {group_row['GroupStartTime']} ns")
                print(f"    Group End Time:          {group_row['GroupEndTime']} ns")
                print(f"    Group Duration:          {group_row['GroupDuration (ns)']} ns")
        else:
            print("  No paths found for this pair.")
        print("\n" + "-" * 70)
else:
    print("Packet tracking DataFrame is empty.")

In [ ]:
pd.set_option('display.max_rows', 600)

In [ ]:
packet_tracking_df[(packet_tracking_df['OriginalSrc'] == 2)& (packet_tracking_df['FinalDst'] == 4)].head(50)

In [1]:
import pandas as pd
from synthesize_results import synthesize
def process_and_sort_results(df):
    """Process the DataFrame to merge ns3 results and sort."""
    if df.empty:
        return df
    
    # Drop columns that are not needed for joining
    df = df.drop(columns=['tag', 'workload_node_id'])
    
    # Group by the message identifiers and aggregate the data
    # This will merge rows for ns3 with the other models
    group_cols = ['src', 'dst', 'chunk_id']
    df = df.groupby(group_cols).sum().reset_index()
    
    # Sort by g2 start time if the column exists
    if 'start_time_g2' in df.columns:
        df = df.sort_values(by='start_time_g2').reset_index(drop=True)
        
    return df

# List of run folders to analyze
run_folders = [
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_101154",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_101517",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_101856",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_102253",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_102624",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_102931",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_103301",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_103615",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_103924",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_104213",
    # "/home/xavid/feina/astra-sim/upc/comparison_run/FoldedClos/all_gather/run_20251021_141336",
    "/app/astra-sim/upc/output/comparison_run/FoldedClos/toy_all_to_all_one_collective/run_20251115_014747"
    # Add other run folder paths here
]

all_results = []
for folder in run_folders:
    # Synthesize results for each run
    results_df = synthesize(folder)
    # Process and sort the DataFrame for the current run
    processed_df = process_and_sort_results(results_df)
    all_results.append(processed_df)

# Concatenate all results into a single DataFrame
if all_results:
    combined_df = pd.concat(all_results)
    
    # Group by the identifiers and calculate the mean across all runs
    group_cols = ['src', 'dst', 'chunk_id']
    averaged_df = combined_df.groupby(group_cols).mean().reset_index()
    
    # Optionally, sort the final averaged DataFrame
    if 'start_time_g2' in averaged_df.columns:
        averaged_df = averaged_df.sort_values(by='start_time_g2').reset_index(drop=True)
else:
    averaged_df = pd.DataFrame()


# Muestra el DataFrame resultante
averaged_df

,src,dst,chunk_id,start_time_ns3,send_time_ns3,arrival_time_ns3
0,0,1,0,1.000000e+01,2.037705e+09,2.037705e+09
1,0,1,1,2.045431e+09,2.039017e+09,4.084448e+09
2,0,1,2,4.097525e+09,2.036614e+09,6.134139e+09
3,0,1,3,6.148790e+09,2.024849e+09,8.173639e+09
4,0,10,0,1.000000e+01,2.015635e+09,2.015635e+09
...,...,...,...,...,...,...
955,9,7,3,6.157501e+09,2.036193e+09,8.193694e+09
956,9,8,0,1.000000e+01,2.037763e+09,2.037763e+09
957,9,8,1,2.054370e+09,2.029836e+09,4.084206e+09
958,9,8,2,4.095783e+09,2.035645e+09,6.131428e+09


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Filter the DataFrame for chunk_id == 2 (accept int or str)
chunk_2_df = averaged_df[averaged_df['chunk_id'].isin([2, '2'])]

# Check if the filtered DataFrame is not empty and contains the required columns
if not chunk_2_df.empty and 'arrival_time_g2' in chunk_2_df.columns and 'arrival_time_ns3' in chunk_2_df.columns:
    data = chunk_2_df[['arrival_time_g2', 'arrival_time_ns3']].dropna().astype(float)
    x = data['arrival_time_g2']
    y = data['arrival_time_ns3']

    # compute common symmetric limits with a small margin
    vmin = min(x.min(), y.min())
    vmax = max(x.max(), y.max())
    if np.isfinite(vmin) and np.isfinite(vmax):
        span = vmax - vmin
        if span == 0:
            margin = 0.1 * (abs(vmin) if vmin != 0 else 1.0)
        else:
            margin = 0.05 * span
        lims = (vmin - margin, vmax + margin)
    else:
        lims = (0, 1)

    plt.figure(figsize=(8, 8))
    plt.scatter(x, y, alpha=0.7, label='Chunk 2 Arrivals')
    plt.plot(lims, lims, 'r--', alpha=0.75, zorder=0, label='Ideal Match (y=x)')

    plt.xlim(lims)
    plt.ylim(lims)

    ax = plt.gca()
    ax.set_aspect('equal', adjustable='box')

    plt.xlabel("Arrival Time G2 (ns)")
    plt.ylabel("Arrival Time NS3 (ns)")
    plt.title("Comparison of Arrival Times (G2 vs. NS3) for Chunk ID 2")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("No data to plot for chunk_id=2, or 'arrival_time_g2'/'arrival_time_ns3' columns are missing.")

In [11]:
import os
import pandas as pd
import re
from functools import reduce
from typing import List, Optional, Dict

def find_config_file(folder_path: str) -> Optional[str]:
    """
    Finds a configuration file (ending with .txt) within the 'configs' subfolder.
    Returns the full path to the first .txt file found, or None.
    """
    config_dir = os.path.join(folder_path, 'configs')
    if not os.path.isdir(config_dir):
        return None
    for item in os.listdir(config_dir):
        if item.endswith('.txt'):
            return os.path.join(config_dir, item)
    return None

def parse_config(file_path: str) -> Dict[str, str]:
    """
    Parses a simple 'key value' or 'key = value' configuration file into a dictionary.
    Converts keys to lowercase.
    """
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                if '=' in line:
                    parts = line.split('=', 1)
                else:
                    parts = line.split(None, 1)
                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except FileNotFoundError:
        print(f"Config file not found: {file_path}")
    except Exception as e:
        print(f"Error parsing config file {file_path}: {e}")
    return params

def parse_ns3_astrasim_fct(path: str) -> list[dict[str, any]]:
    """
    Parse ns3/astrasim_fct.txt lines.
    Returns a list of dictionaries, each representing a message.
    """
    results = []
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            for ln in f:
                ln = ln.strip()
                if not ln:
                    continue
                parts = ln.split()
                if len(parts) < 8:
                    continue
                try:
                    src = str(int(parts[0][-4:-2], 16))
                    dst = str(int(parts[1][-4:-2], 16))
                    chunk_id = str(int(parts[2]) - 10000)
                    issue_tick = float(parts[-3])
                    delay = float(parts[-2])
                    results.append({
                        "src": src,
                        "dst": dst,
                        "chunk_id": chunk_id,
                        "arrival_time": issue_tick+delay
                    })
                except (ValueError, IndexError):
                    pass
    except FileNotFoundError:
        print(f"FCT file not found: {path}")
    return results

# --- Main Logic ---
base_run_folder = '/app/astra-sim/upc/output/comparison_run/FoldedClos/toy_all_to_all_one_collective'
run_folders = [os.path.join(base_run_folder, d) for d in os.listdir(base_run_folder) if os.path.isdir(os.path.join(base_run_folder, d))]

all_run_dfs = []
run_name_counts = {}

for folder in run_folders:
    print(f"Processing: {os.path.basename(folder)}")

    # 1. Create descriptive name from config
    config_file = find_config_file(folder)
    if not config_file:
        print(f"  - Skipping: Config file not found.")
        continue
    config_params = parse_config(config_file)
    
    # Note: Keys are lowercased by the parser
    base_run_name = (
        f"cc:{config_params.get('cc_mode', 'N/A')},"
        f"win:{config_params.get('has_win', 'N/A')},"
        f"adapt:{config_params.get('var_win', 'N/A')},"
        f"buf:{config_params.get('buffer_size', 'N/A')},"
        f"size:{config_params.get('packet_payload_size', 'N/A')}"
    )

    # Handle duplicate run names by appending a suffix
    count = run_name_counts.get(base_run_name, 0)
    run_name_counts[base_run_name] = count + 1
    run_name = f"{base_run_name}_{count}" if count > 0 else base_run_name


    # 2. Parse NS3 FCT data
    fct_file = os.path.join(folder, 'ns3', 'astrasim_fct.txt')
    ns3_data = parse_ns3_astrasim_fct(fct_file)
    
    if not ns3_data:
        print(f"  - Skipping: No data in {fct_file}")
        continue
        
    df = pd.DataFrame(ns3_data)
    df = df.rename(columns={'arrival_time': run_name})
    all_run_dfs.append(df)

# 3. Merge all DataFrames into one
if all_run_dfs:
    # Use reduce to iteratively merge all dataframes on the message identifiers
    id_cols = ['src', 'dst', 'chunk_id']
    merged_df = reduce(lambda left, right: pd.merge(left, right, on=id_cols, how='outer'), all_run_dfs)
    
    # Sort for readability
    merged_df['src'] = pd.to_numeric(merged_df['src'])
    merged_df['dst'] = pd.to_numeric(merged_df['dst'])
    merged_df['chunk_id'] = pd.to_numeric(merged_df['chunk_id'])
    merged_df = merged_df.sort_values(by=id_cols).reset_index(drop=True)

else:
    merged_df = pd.DataFrame()

# Display the final DataFrame
merged_df

Processing: run_20251115_155016
Processing: run_20251115_160005
  - Skipping: No data in /app/astra-sim/upc/output/comparison_run/FoldedClos/toy_all_to_all_one_collective/run_20251115_160005/ns3/astrasim_fct.txt
Processing: run_20251115_155605
  - Skipping: No data in /app/astra-sim/upc/output/comparison_run/FoldedClos/toy_all_to_all_one_collective/run_20251115_155605/ns3/astrasim_fct.txt
Processing: run_20251115_160605
  - Skipping: No data in /app/astra-sim/upc/output/comparison_run/FoldedClos/toy_all_to_all_one_collective/run_20251115_160605/ns3/astrasim_fct.txt
Processing: run_20251115_153910
Processing: run_20251115_154210
Processing: run_20251115_160205
  - Skipping: No data in /app/astra-sim/upc/output/comparison_run/FoldedClos/toy_all_to_all_one_collective/run_20251115_160205/ns3/astrasim_fct.txt
Processing: run_20251115_154411
Processing: run_20251115_153810
Processing: run_20251115_154713
Processing: run_20251115_153710
Processing: run_20251115_154011
Processing: run_20251115

,src,dst,chunk_id,"cc:10,win:1,adapt:1,buf:32,size:1500","cc:3,win:1,adapt:1,buf:1,size:1500","cc:3,win:1,adapt:1,buf:1,size:1500_1","cc:3,win:1,adapt:1,buf:32,size:1500","cc:3,win:1,adapt:1,buf:32,size:1500_1","cc:3,win:1,adapt:1,buf:32,size:1500_2","cc:3,win:1,adapt:1,buf:8,size:1500",...,"cc:3,win:1,adapt:1,buf:8,size:1500_2","cc:8,win:1,adapt:1,buf:32,size:1500","cc:3,win:1,adapt:1,buf:8,size:1500_3","cc:10,win:1,adapt:1,buf:8,size:1500","cc:8,win:1,adapt:1,buf:1,size:1500","cc:3,win:1,adapt:1,buf:32,size:1500_3","cc:3,win:1,adapt:1,buf:1,size:1500_2","cc:10,win:1,adapt:1,buf:1,size:1500","cc:8,win:1,adapt:1,buf:8,size:1500","cc:3,win:1,adapt:1,buf:1,size:1500_3"
0,0,1,0,6.180204e+09,8.138722e+09,8.136954e+09,8.136954e+09,8.136954e+09,8.140666e+09,8.136954e+09,...,8.140666e+09,6.853527e+09,8.136954e+09,6.180204e+09,6.853527e+09,8.138722e+09,8.136954e+09,6.180204e+09,6.853527e+09,8.140666e+09
1,0,2,0,7.919812e+09,8.121294e+09,8.116407e+09,8.116407e+09,8.116407e+09,8.112058e+09,8.116407e+09,...,8.112058e+09,7.974927e+09,8.116407e+09,7.919812e+09,7.974927e+09,8.121294e+09,8.116407e+09,7.919812e+09,7.974927e+09,8.112058e+09
2,0,3,0,7.870140e+09,8.121332e+09,8.137301e+09,8.137301e+09,8.137301e+09,8.112134e+09,8.137301e+09,...,8.112134e+09,7.962279e+09,8.137301e+09,7.870140e+09,7.962279e+09,8.121332e+09,8.137301e+09,7.870140e+09,7.962279e+09,8.112134e+09
3,0,4,0,8.490163e+09,8.005161e+09,8.118522e+09,8.118522e+09,8.118522e+09,7.960256e+09,8.118522e+09,...,7.960256e+09,8.529975e+09,8.118522e+09,8.490163e+09,8.529975e+09,8.005161e+09,8.118522e+09,8.490163e+09,8.529975e+09,7.960256e+09
4,0,5,0,8.497457e+09,8.136266e+09,8.138589e+09,8.138589e+09,8.138589e+09,8.138898e+09,8.138589e+09,...,8.138898e+09,8.556783e+09,8.138589e+09,8.497457e+09,8.556783e+09,8.136266e+09,8.138589e+09,8.497457e+09,8.556783e+09,8.138898e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,15,10,0,8.432221e+09,8.138418e+09,8.139083e+09,8.139083e+09,8.139083e+09,8.143928e+09,8.139083e+09,...,8.143928e+09,8.642746e+09,8.139083e+09,8.432221e+09,8.642746e+09,8.138418e+09,8.139083e+09,8.432221e+09,8.642746e+09,8.143928e+09
236,15,11,0,8.344221e+09,8.135248e+09,8.138893e+09,8.138893e+09,8.138893e+09,8.141661e+09,8.138893e+09,...,8.141661e+09,8.642698e+09,8.138893e+09,8.344221e+09,8.642698e+09,8.135248e+09,8.138893e+09,8.344221e+09,8.642698e+09,8.141661e+09
237,15,12,0,7.922118e+09,8.138803e+09,8.138470e+09,8.138470e+09,8.138470e+09,8.147592e+09,8.138470e+09,...,8.147592e+09,8.068647e+09,8.138470e+09,7.922118e+09,8.068647e+09,8.138803e+09,8.138470e+09,7.922118e+09,8.068647e+09,8.147592e+09
238,15,13,0,8.005265e+09,8.143865e+09,8.138969e+09,8.138969e+09,8.138969e+09,8.143429e+09,8.138969e+09,...,8.143429e+09,8.030415e+09,8.138969e+09,8.005265e+09,8.030415e+09,8.143865e+09,8.138969e+09,8.005265e+09,8.030415e+09,8.143429e+09
